# PrimeVul Qwen unseen-project evaluation candidate

This notebook runs the frozen Thesis LLM-only baseline on the new PrimeVul unseen-project holdout candidate. It is designed for direct upload to Google Colab.

It uses code-only prompts, the leakage-controlled manifest, deterministic greedy decoding, and the pinned `Qwen/Qwen2.5-Coder-1.5B-Instruct` model at revision `2e1fd39`. The candidate is used only for independent evaluation; do not refit the policy from these outputs: do not fit calibration, select thresholds, or change the operating policy from these outputs.

The default run processes three retained test records as a smoke test. After it succeeds, set `SAMPLE_COUNT = 0` in the configuration cell and rerun from that cell onward to process all retained test records.

In [ ]:
# Colab normally includes torch. Install the runtime packages used by the baseline.
!pip -q install -U transformers accelerate safetensors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import re
import time

PROJECT_DIR = Path('/content/drive/MyDrive/primevul')

def find_drive_file(filename):
    direct = PROJECT_DIR / filename
    if direct.exists():
        return direct
    matches = list(Path('/content/drive/MyDrive').rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Could not find {filename!r} under /content/drive/MyDrive')
    if len(matches) > 1:
        print(f'Multiple copies of {filename} found; using {matches[0]}')
    return matches[0]

DATA_PATH = find_drive_file('primevul_train.jsonl')
MANIFEST_PATH = find_drive_file('primevul_unseen_project_holdout_manifest.json')

MODEL_SOURCE = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
MODEL_REVISION = '2e1fd39'
USE_LOCAL_MODEL = False  # Set True and change MODEL_SOURCE if using a local Drive folder.
PARTITION = 'unseen_project_holdout'
SAMPLE_COUNT = 3  # Set to 0 for all retained unseen-project records.
SEED = 12345
MAX_NEW_TOKENS = 16
MAX_INPUT_TOKENS = 4096
OOM_RETRY_INPUT_TOKENS = 2048
RESUME_FULL_RUN = True
OUTPUT_PATH = PROJECT_DIR / ('qwen_smoke_unseen_project.json' if SAMPLE_COUNT else 'qwen_unseen_project_results.jsonl')

def load_retained_numbers(manifest_path, partition):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('status') != 'complete':
        raise ValueError(f'Manifest is not complete: {manifest_path}')
    part = manifest.get('partitions', {}).get(partition)
    if not part:
        raise ValueError(f'Partition {partition!r} not found in manifest')
    return set(part['kept_record_numbers'])

def iter_records(data_path):
    with data_path.open('r', encoding='utf-8', errors='replace') as handle:
        for record_number, line in enumerate(handle, start=1):
            if line.strip():
                yield record_number, json.loads(line)

retained_numbers = load_retained_numbers(MANIFEST_PATH, PARTITION)
if SAMPLE_COUNT < 0:
    raise ValueError('SAMPLE_COUNT must be non-negative')
if SAMPLE_COUNT == 0:
    selected_numbers = None
else:
    available = [n for n, _ in iter_records(DATA_PATH) if n in retained_numbers]
    if not available:
        raise ValueError('No retained test records were found in the data file')
    selected_numbers = set(random.Random(SEED).sample(available, min(SAMPLE_COUNT, len(available))))
expected_count = len(retained_numbers) if selected_numbers is None else len(selected_numbers)
print('Data:', DATA_PATH)
print('Manifest:', MANIFEST_PATH)
print('Model:', MODEL_SOURCE)
print('Retained unseen-project records in manifest:', len(retained_numbers))
print('Records selected for this run:', expected_count)
print('Output:', OUTPUT_PATH)
if selected_numbers is not None:
    print('Selected record numbers:', sorted(selected_numbers))

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

local_model = USE_LOCAL_MODEL or Path(MODEL_SOURCE).expanduser().exists()
load_kwargs = {'trust_remote_code': False, 'local_files_only': local_model}
if not local_model:
    load_kwargs['revision'] = MODEL_REVISION
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, **load_kwargs)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_kwargs = dict(load_kwargs)
model_kwargs.update({'torch_dtype': dtype, 'low_cpu_mem_usage': True, 'device_map': 'auto'})
model = AutoModelForCausalLM.from_pretrained(MODEL_SOURCE, **model_kwargs)
model.eval()
input_device = model.get_input_embeddings().weight.device
print('Input device:', input_device)
print('Loaded dtype:', dtype)

In [ ]:
SYSTEM_PROMPT = 'You are a security expert that is good at static program analysis.'
USER_TEMPLATE = '''Please analyze the following code:
```
{func}
```
Please indicate your analysis result with one of the options:
(1) YES: A security vulnerability detected.
(2) NO: No security vulnerability.

Only reply with one of the options above. Do not include any further information.
'''
LABEL_RE = re.compile(r'\b(YES|NO)\b', re.IGNORECASE)

def prompt_messages(code):
    return [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': USER_TEMPLATE.format(func=code)}]

def prompt_hash(code):
    return hashlib.sha256(json.dumps(prompt_messages(code), sort_keys=True).encode('utf-8')).hexdigest()

def parse_prediction(text):
    match = LABEL_RE.search(text or '')
    return match.group(1).upper() if match else None

def top_token_probabilities(scores):
    probabilities = torch.softmax(scores.float(), dim=-1)
    values, indices = torch.topk(probabilities, k=min(5, probabilities.shape[-1]))
    return [{'token_id': int(i), 'token': tokenizer.decode([int(i)]), 'probability': float(p)} for p, i in zip(values.tolist(), indices.tolist())]

def prepare_inputs(code, max_input_tokens):
    messages = prompt_messages(code)
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
    full_length = inputs['input_ids'].shape[-1]
    truncated = full_length > max_input_tokens
    if truncated:
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, truncation=True, max_length=max_input_tokens, return_dict=True, return_tensors='pt')
    inputs = {key: value.to(input_device) for key, value in inputs.items()}
    return inputs, truncated, int(inputs['input_ids'].shape[-1])

def run_one(record, max_input_tokens):
    code = str(record.get('func', ''))
    inputs, truncated, input_tokens = prepare_inputs(code, max_input_tokens)
    with torch.inference_mode():
        outputs = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, return_dict_in_generate=True, output_scores=True, pad_token_id=tokenizer.pad_token_id)
    input_length = inputs['input_ids'].shape[-1]
    generated_ids = outputs.sequences[0][input_length:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return {'record': {'idx': record.get('idx'), 'target': record.get('target'), 'project': record.get('project'), 'func_hash': record.get('func_hash')}, 'response': response, 'prediction': parse_prediction(response), 'first_token_top5': top_token_probabilities(outputs.scores[0][0]), 'prompt_sha256': prompt_hash(code), 'input_tokens': input_tokens, 'input_token_limit': max_input_tokens, 'input_truncated': truncated}

In [ ]:
# Stream the full unseen-project run to Drive. Existing JSONL records are resumed when possible.
started = time.time()
processed_this_run = 0
results = []
jsonl_mode = SAMPLE_COUNT == 0
completed_numbers = set()
if jsonl_mode and RESUME_FULL_RUN and OUTPUT_PATH.exists():
    with OUTPUT_PATH.open(encoding='utf-8') as existing:
        for line in existing:
            try:
                number = json.loads(line).get('record_number')
                if isinstance(number, int):
                    completed_numbers.add(number)
            except json.JSONDecodeError:
                pass
print('Records already completed:', len(completed_numbers))
print('Records remaining:', expected_count - len(completed_numbers))
output_handle = OUTPUT_PATH.open('a' if completed_numbers else 'w', encoding='utf-8') if jsonl_mode else None
try:
    for record_number, record in iter_records(DATA_PATH):
        if record_number not in retained_numbers or (selected_numbers is not None and record_number not in selected_numbers) or record_number in completed_numbers:
            continue
        processed_this_run += 1
        print(f'[{len(completed_numbers) + processed_this_run}/{expected_count}] record {record_number} (idx={record.get("idx")})', flush=True)
        oom_retry = False
        try:
            result = run_one(record, MAX_INPUT_TOKENS)
        except torch.cuda.OutOfMemoryError:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            result = run_one(record, OOM_RETRY_INPUT_TOKENS)
            oom_retry = True
        result['oom_retry'] = oom_retry
        result['record_number'] = record_number
        if output_handle is not None:
            output_handle.write(json.dumps(result, ensure_ascii=False) + '\n')
            output_handle.flush()
        else:
            results.append(result)
        completed_numbers.add(record_number)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
finally:
    if output_handle is not None:
        output_handle.close()
if len(completed_numbers) != expected_count:
    raise RuntimeError(f'Completed {len(completed_numbers)} records, expected {expected_count}')
metadata = {'status': 'complete', 'model': {'source': MODEL_SOURCE, 'revision': None if local_model else MODEL_REVISION, 'dtype': str(dtype), 'device': str(input_device)}, 'run': {'partition': PARTITION, 'records_processed': len(completed_numbers), 'records_processed_this_run': processed_this_run, 'retained_manifest_records': len(retained_numbers), 'sample_count': SAMPLE_COUNT, 'seed': SEED, 'max_new_tokens': MAX_NEW_TOKENS, 'max_input_tokens': MAX_INPUT_TOKENS, 'oom_retry_input_tokens': OOM_RETRY_INPUT_TOKENS, 'resume_full_run': RESUME_FULL_RUN, 'prompt_contract': 'code-only YES/NO classification', 'data': str(DATA_PATH), 'manifest': str(MANIFEST_PATH), 'elapsed_seconds': round(time.time() - started, 3), 'output_format': 'jsonl' if jsonl_mode else 'json'}}
if not jsonl_mode:
    OUTPUT_PATH.write_text(json.dumps({**metadata, 'results': results}, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
metadata_path = Path(str(OUTPUT_PATH) + '.meta.json')
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Completed records:', len(completed_numbers))
print('Results:', OUTPUT_PATH)
print('Metadata:', metadata_path)

In [ ]:
if not jsonl_mode:
    saved = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
    for item in saved['results']:
        print('Record:', item['record'])
        print('Prediction:', item['prediction'])
        print('Response:', repr(item['response']))
        print('First-token top-5:', item['first_token_top5'])
        print('-' * 80)
else:
    print('Full-run JSONL was written to:', OUTPUT_PATH)
    with OUTPUT_PATH.open(encoding='utf-8') as handle:
        for _ in range(3):
            line = handle.readline()
            if not line:
                break
            print(line.rstrip())

## What to do after the smoke test

Confirm that the smoke run completes and produces `qwen_smoke_unseen_project.json` plus its metadata sidecar. Then change `SAMPLE_COUNT = 0` and rerun from the configuration cell onward. The expected complete retained candidate count is 38,362.

The resulting `qwen_unseen_project_results.jsonl` and `qwen_unseen_project_results.jsonl.meta.json` are raw model outputs for the unseen-project evaluation candidate. Send or upload both files for the next frozen-policy and static-corroboration steps. Do not use candidate labels to refit temperature/Platt scaling or choose thresholds.